# Module 01 — Text Generation

The basics of the **Messages API**: single-turn calls, system prompts, multi-turn conversations, generation parameters, streaming, and prefill.

This builds on **Module 00** and uses the same **`AnthropicVertex` / ADC** path — so every call here is also captured by the **BigQuery logging** you enabled in Module 00.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging). Here we just install, auto-detect the project, and build `client`.

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Core text generation

### Basic single-turn message

The simplest call: one user message in, one assistant reply out. The response text lives in `message.content[0].text`.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "Explain what an API is in two sentences."}],
)
print(message.content[0].text)

### System prompts

The `system` parameter steers behavior, persona, and output format. It's separate from the conversation `messages` and applies to the whole call.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system=(
        "You are a concise technical summarizer. Reply in at most three bullet "
        "points, plain language, no preamble."
    ),
    messages=[{
        "role": "user",
        "content": "Summarize what a load balancer does and why teams use one.",
    }],
)
print(message.content[0].text)

### Multi-turn conversation

A conversation is a list of messages alternating `user` / `assistant`. The API is **stateless** — there's no server-side memory, so you **resend the full prior history** on every call. To continue, append the new `user` turn after the previous `assistant` turn.

In [ ]:
messages = [
    {"role": "user", "content": "Suggest a name for a tool that tracks cloud costs."},
    {"role": "assistant", "content": "How about \"CloudLedger\"?"},
    {"role": "user", "content": "Good. Now give me a one-line tagline for it."},
]

message = client.messages.create(model=MODEL, max_tokens=256, messages=messages)
print(message.content[0].text)

### Generation parameters

- **`max_tokens`** — hard cap on output length (required).
- **`temperature`** — randomness, `0.0`–`1.0`. Low = focused/deterministic, high = varied/creative.
- **`top_p`** — nucleus sampling; an alternative to temperature. Tune one, not both.
- **`stop_sequences`** — strings that halt generation when produced; the stop text itself is not included in the output.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    temperature=0.2,
    stop_sequences=["3."],   # stop as soon as the model starts item 3
    messages=[{"role": "user", "content": "List steps to deploy a web app. Number them."}],
)
print(message.content[0].text)
print("-" * 60)
# stop_reason == "stop_sequence" means a stop string ended generation.
print(f"stop_reason: {message.stop_reason}")

## Part D — Streaming

Streaming returns tokens **incrementally** as they're generated instead of waiting for the full response — better perceived latency and a live-typing UX. Use the `client.messages.stream(...)` context manager and iterate over `stream.text_stream`.

In [ ]:
with client.messages.stream(
    model=MODEL,
    max_tokens=512,
    messages=[{"role": "user", "content": "Write a short paragraph about why observability matters."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="")

## Part E — Useful patterns

### Prefill

End your `messages` with an **`assistant`** turn that begins the output you want — the model **continues from it** rather than repeating it. This is a reliable way to force a format (JSON, a table, a specific opening) or skip preamble. The returned text is the *continuation*, so prepend your prefill if you need the complete string.

In [ ]:
prefill = "{\n  \"service\":"
message = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "Give me a JSON object describing a web service: service, port, protocol."},
        {"role": "assistant", "content": prefill},   # seed the start of the answer
    ],
)
# Response continues the prefill — recombine for the full object.
print(prefill + message.content[0].text)

### Inspecting the response

Beyond the text, every response carries **usage** (token counts for cost/attribution), a **`stop_reason`**, and a list of **content blocks** (`message.content`).

In [ ]:
print(f"input_tokens : {message.usage.input_tokens}")
print(f"output_tokens: {message.usage.output_tokens}")
print(f"stop_reason  : {message.stop_reason}")
# stop_reason: "end_turn" = finished naturally; "max_tokens" = hit the cap;
# "stop_sequence" = a stop string was emitted.

## Part F — Recap

- **Single-turn:** one `user` message → `message.content[0].text`.
- **System prompts** steer persona/format via the `system` param.
- **Multi-turn** is a stateless, alternating `user`/`assistant` list — resend history each call.
- **Parameters:** `max_tokens`, `temperature`, `top_p`, `stop_sequences` (watch `stop_reason`).
- **Streaming** via `client.messages.stream(...)` and `stream.text_stream` for incremental output.
- **Prefill** seeds the assistant turn to force a format; the reply continues it.
- **Inspect** `usage` tokens, `stop_reason`, and content blocks on every response.

**Next: Module 02 — Vision (image input).**